# Hyperedges

A hyperedge connects any number of vertices, not just two. Two flavours:

- **Undirected** — one set of *members*. Useful for complexes, sets,
  motifs.
- **Directed** — a *head* set and a *tail* set. Useful for reactions
  (substrates → products), causal motifs.


In [ ]:
from annnet import AnnNet

H = AnnNet(directed=False)
H.add_vertices(['Glc', 'ATP', 'G6P', 'ADP', 'F6P'])
H


## Undirected hyperedges

Pass a list of members. Order doesn't matter.


In [ ]:
H.add_edges(['Glc', 'ATP', 'G6P'], edge_id='complex_A')
H.add_edges(['G6P', 'F6P'], edge_id='complex_B')  # 2-member is still a hyperedge here

# Inspect the members
rec = H._edges['complex_A']
print('complex_A members:', set(rec.src))
print('  directed?      ', rec.directed)


## Directed hyperedges (reactions)

Pass `src=[heads]` and `tgt=[tails]`. The head set produces the tail set
(or with the opposite convention — pick once and be consistent).


In [ ]:
# Hexokinase: Glc + ATP -> G6P + ADP
H.add_edges(src=['Glc', 'ATP'], tgt=['G6P', 'ADP'], edge_id='hexokinase')

rec = H._edges['hexokinase']
print('hexokinase head (src):', set(rec.src))
print('hexokinase tail (tgt):', set(rec.tgt))
print('  directed?           ', rec.directed)


## Reading hyperedges via views

`G.views.edges()` exposes hyperedges through the `members` column (for
undirected) or `head` / `tail` columns (for directed). The `kind` column
is `'hyper'` for both.


In [ ]:
H.views.edges().select(['edge_id', 'kind', 'members', 'head', 'tail']).head()


## Stoichiometric coefficients (advanced)

The incidence matrix is the source of truth. Each column is one edge;
each row is one entity. For a binary edge in a directed graph the
convention is `+w` at source, `-w` at target. For a directed hyperedge
the same rule applies per member of head / tail (positive on head,
negative on tail by default).

You can set custom coefficients per member via `G.set_edge_coeffs`.


In [ ]:
# Read the column for hexokinase straight off the matrix
import numpy as np

eid = 'hexokinase'
col = H._edges[eid].col_idx
M = H._matrix

print(f'incidence column for {eid}:')
for vid in H.vertices():
    rec_row = H._entities[H._resolve_entity_key(vid)].row_idx
    val = M[rec_row, col]
    if val != 0:
        print(f'  {vid:>5}: {val:+.2f}')


---

Next: [06 — Multilayer](06_multilayer.ipynb).
